# Data Cleaning

## Module Imports

In [1]:
import numpy as np 
import pandas as pd 

## customers data cleaning

In [2]:
data = pd.read_csv("./Raw_Data/customers.csv")

In [3]:
data.head()

,customer_id,full_name,email,phone,gender,registration_date,city_id,birth_year
0,C000001,Nida Memon,nida.memon6133@yahoo.com,0343-4328042,F,2024-02-04,614.0,1996.0
1,C000002,rizwan ansari,rizwan.ansari7047@gmail.com,03292755530,m,03/02/2025,310.0,1977.0
2,C000003,Noor Baig,noor.baig763@gmail.com,330 2812162,F,2024-01-31,347.0,1999.0
3,C000004,Naveed Hashmi,naveed.hashmi9719@gmail.com,03128373694,M,2024-07-18,570.0,1985.0
4,C000005,Mahnoor Mirza,mahnoor.mirza5501@yahoo.com,03251346045,F,2024-04-07,614.0,1989.0


In [4]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45000 entries, 0 to 44999
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   customer_id        45000 non-null  object 
 1   full_name          45000 non-null  object 
 2   email              43302 non-null  object 
 3   phone              41401 non-null  object 
 4   gender             42801 non-null  object 
 5   registration_date  45000 non-null  object 
 6   city_id            42767 non-null  float64
 7   birth_year         40400 non-null  float64
dtypes: float64(2), object(6)
memory usage: 2.7+ MB


In [14]:
# duplicate id check
data["customer_id"].duplicated().sum()

np.int64(0)

In [ ]:
# Full name format cleaning
data['full_name'] = (
    data['full_name']
    .str.strip()
    .str.replace(r'\s+', ' ', regex=True)
    .str.title()
)

In [ ]:
# Gender values check
print(data["gender"].value_counts())

gender
M         16672
F         13344
male       2382
Male       2328
m          2327
f          1949
Female     1909
female     1890
Name: count, dtype: int64


In [ ]:
# gender value format cleaning
data['gender'] = data['gender'].astype(str).str.strip().str[0].str.upper()

In [16]:
# phone number format cleaning
# 1. Extract only numeric digits
digits = data['phone'].astype(str).str.replace(r'\D', '', regex=True)

# 2. Take the last 10 digits (the local subscriber number) and prepend '0'
clean_11 = '0' + digits.str[-10:]

# 3. Format as '03xx xxxxxxx'
formatted = clean_11.str[:4] + ' ' + clean_11.str[4:]

# 4. Keep true NaN / null for missing or invalid numbers
data['phone'] = formatted.where(digits.str.len() >= 10, np.nan)

In [17]:
# registration data format cleaning
# 'dayfirst=True' ensures formats like 14/04/2024 and 03/02/2025 are parsed correctly
data['registration_date'] = pd.to_datetime(
    data['registration_date'], 
    format='mixed', 
    dayfirst=True, 
    errors='coerce'
)

data['registration_date'] = data['registration_date'].dt.strftime('%Y-%m-%d')

In [18]:
# for city id 
# Convert to nullable integer (handles numbers cleanly without converting to 614.0)
data['city_id'] = pd.to_numeric(data['city_id'], errors='coerce').astype('Int64')

# flag missing cities explicitly with a distinct ID:
data['city_id'] = data['city_id'].fillna(-1)

In [21]:
# for birth year
data['birth_year'] = pd.to_numeric(data['birth_year'], errors='coerce').astype('Int64')

# Check for invalid birth year ranges (e.g., < 1920 or > current year)
invalid_years = data[(data['birth_year'] < 1920) | (data['birth_year'] > 2026)]
print(f"Suspicious birth years: {len(invalid_years)}")

# Impute missing values with the median birth year:
median_year = int(data['birth_year'].median())
data['birth_year'] = data['birth_year'].fillna(median_year)

#removing inconsistencies
data[data['birth_year'] == 142] = median_year

Suspicious birth years: 142


In [22]:
# saving data

data.to_csv('./Cleaned_Data/customers_cleaned.csv', index=False)

## Locations data cleaning


In [23]:
# Load locations
locations = pd.read_csv('./Raw_data/locations.csv')

In [24]:
# 1. Clean and standardize column names
locations.columns = locations.columns.str.strip().str.lower()

In [25]:
# 2. Standardize City casing
locations['city'] = locations['city'].astype(str).str.strip().str.title()

In [26]:
# 3. Standardize Province names
province_mapping = {
    'kp': 'Khyber Pakhtunkhwa',
    'khyber pakhtunkhwa': 'Khyber Pakhtunkhwa',
    'baluchistan': 'Balochistan',
    'balochistan': 'Balochistan',
    'sindh': 'Sindh',
    'punjab': 'Punjab',
    'gilgit-baltistan': 'Gilgit-Baltistan',
    'islamabad': 'Islamabad Capital Territory',
    'ajk': 'Azad Jammu and Kashmir'
}

In [27]:
# Normalize text before mapping
locations['province'] = (
    locations['province']
    .astype(str)
    .str.strip()
    .str.lower()
    .map(lambda x: province_mapping.get(x, x.title()))
)

In [28]:
# 4. Standardize Region & Population Tier
locations['region'] = locations['region'].astype(str).str.strip().str.title()
locations['population_tier'] = locations['population_tier'].astype(str).str.strip()

In [29]:
# 5. Convert coordinates to numeric (float), converting blanks to NaN
locations['latitude'] = pd.to_numeric(locations['latitude'], errors='coerce')
locations['longitude'] = pd.to_numeric(locations['longitude'], errors='coerce')

In [30]:
# 6. Ensure city_id is integer
locations['city_id'] = pd.to_numeric(locations['city_id'], errors='coerce').astype('Int64')

In [31]:
# 7. Check for anomalies / summary
print("Missing coordinate counts:")
print(locations[['latitude', 'longitude']].isna().sum())

Missing coordinate counts:
latitude     33
longitude    31
dtype: int64


In [32]:
print("\nUnique Provinces after cleaning:")
print(locations['province'].value_counts())


Unique Provinces after cleaning:
province
Punjab                         273
Sindh                          141
Khyber Pakhtunkhwa              94
Balochistan                     56
Nan                             25
Panjab                          17
Gilgit-Baltistan                13
Azad Kashmir                    13
Sind                             9
Kpk                              4
Gb                               2
Gilgit Baltistan                 1
Islamabad Capital Territory      1
Azad Jammu & Kashmir             1
Name: count, dtype: int64


In [34]:
# 8. Save cleaned file
locations.to_csv('./Cleaned_Data/locations_cleaned.csv', index=False)

## Order Items Data cleaning

In [36]:
# Load order_items
order_items = pd.read_csv('./Raw_Data/order_items.csv')

In [37]:
# 1. Standardize column names
order_items.columns = order_items.columns.str.strip().str.lower()

In [38]:
# 2. Clean string IDs (strip whitespace)
id_cols = ['order_item_id', 'order_id', 'product_id', 'seller_id']
for col in id_cols:
    order_items[col] = order_items[col].astype(str).str.strip()

In [39]:
# 3. Clean quantity (convert to integer, handling potential floats/nulls)
order_items['quantity'] = (
    pd.to_numeric(order_items['quantity'], errors='coerce')
    .fillna(1)
    .astype(int)
)

In [40]:
# 4. Clean unit_price (strip 'Rs.', spaces, commas, then convert to float)
order_items['unit_price'] = (
    order_items['unit_price']
    .astype(str)
    .str.replace(r'[^\d.]', '', regex=True)  # removes 'Rs.', spaces, and commas
)
order_items['unit_price'] = pd.to_numeric(order_items['unit_price'], errors='coerce')

In [41]:
# 5. Clean discount_amount (fill missing with 0, convert to float)
order_items['discount_amount'] = (
    order_items['discount_amount']
    .astype(str)
    .str.replace(r'[^\d.]', '', regex=True)
)
order_items['discount_amount'] = (
    pd.to_numeric(order_items['discount_amount'], errors='coerce')
    .fillna(0.0)
)

In [42]:
# 6. (Optional Best Practice) Add calculated total_price
# total_price = (quantity * unit_price) - discount_amount
order_items['total_price'] = (
    (order_items['quantity'] * order_items['unit_price']) - order_items['discount_amount']
).round(2)

In [43]:
# 7. Verification checks
print("Data Types & Null counts:")
order_items.info()

Data Types & Null counts:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 315143 entries, 0 to 315142
Data columns (total 8 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   order_item_id    315143 non-null  object 
 1   order_id         315143 non-null  object 
 2   product_id       315143 non-null  object 
 3   seller_id        315143 non-null  object 
 4   quantity         315143 non-null  int64  
 5   unit_price       315143 non-null  float64
 6   discount_amount  315143 non-null  float64
 7   total_price      315143 non-null  float64
dtypes: float64(3), int64(1), object(4)
memory usage: 19.2+ MB


In [44]:
print("\nSample cleaned rows:")
print(order_items[['quantity', 'unit_price', 'discount_amount', 'total_price']].head(10))


Sample cleaned rows:
   quantity  unit_price  discount_amount  total_price
0         1       0.229              0.0         0.23
1         1    5850.000              0.0      5850.00
2         1    1900.000              0.0      1900.00
3         1    2180.000              0.0      2180.00
4         1    2100.000              0.0      2100.00
5         1    6340.000              0.0      6340.00
6         2    1840.000              0.0      3680.00
7         2    3570.000              0.0      7140.00
8         1    1340.000            268.0      1072.00
9         1     650.000              0.0       650.00


In [45]:
# 8. Save cleaned file
order_items.to_csv('./Cleaned_Data/order_items_cleaned.csv', index=False)

## Orders Data Cleaning

In [46]:
# 1. Load data ensuring IDs are read as strings (preserves leading zeros)
orders = pd.read_csv('./Raw_Data/orders.csv', dtype={'order_id': str, 'customer_id': str})

In [47]:
# Standardize column headers
orders.columns = orders.columns.str.strip().str.lower()

In [48]:
# 2. Clean order_id & customer_id strings
orders['order_id'] = orders['order_id'].astype(str).str.strip()
orders['customer_id'] = orders['customer_id'].astype(str).str.strip()

In [49]:
# 3. Standardize order_date
orders['order_date'] = pd.to_datetime(
    orders['order_date'], 
    format='mixed', 
    dayfirst=True, 
    errors='coerce'
).dt.strftime('%Y-%m-%d')

In [50]:
# 4. Standardize order_status (fixes typos like 'DELIVERD' and 'Complete')
status_mapping = {
    'deliverd': 'Delivered',
    'delivered': 'Delivered',
    'complete': 'Completed',
    'completed': 'Completed',
    'cancelled': 'Cancelled',
    'returned': 'Returned'
}
orders['order_status'] = (
    orders['order_status']
    .astype(str)
    .str.strip()
    .str.lower()
    .map(lambda x: status_mapping.get(x, x.title()))
)

In [51]:
# 5. Clean ship_city_id (nullable integer)
orders['ship_city_id'] = pd.to_numeric(orders['ship_city_id'], errors='coerce').astype('Int64')

In [52]:
# 6. Clean order_total (strip 'Rs.', commas, convert to float)
orders['order_total'] = (
    orders['order_total']
    .astype(str)
    .str.replace(r'[^\d.]', '', regex=True)
)
orders['order_total'] = pd.to_numeric(orders['order_total'], errors='coerce')

In [53]:
# 7. Standardize discount_code (fill NaN with 'NONE' or keep NaN)
orders['discount_code'] = orders['discount_code'].fillna('NONE').astype(str).str.strip().str.upper()

In [54]:
# 8. Standardize channel
channel_mapping = {
    'web': 'Web',
    'website': 'Web',  # Unifies 'Website' and 'Web' into a single standard channel
    'app': 'App',
    'mobile web': 'Mobile Web'
}
orders['channel'] = (
    orders['channel']
    .astype(str)
    .str.strip()
    .str.lower()
    .map(lambda x: channel_mapping.get(x, x.title()))
)

In [55]:
# 9. Verification Summary
print("--- Cleaned Status Counts ---")
print(orders['order_status'].value_counts())

print("\n--- Cleaned Channels ---")
print(orders['channel'].value_counts())

print("\n--- Missing Values Check ---")
print(orders.isna().sum())

--- Cleaned Status Counts ---
order_status
Delivered     129493
Completed      14445
Cancelled      14024
Returned        2701
In Transit      1206
Canceled         948
Return           210
In-Transit        97
Shipped           76
Name: count, dtype: int64

--- Cleaned Channels ---
channel
App           77324
Web           49012
Mobile Web    31162
Mobile App     4069
Mweb           1633
Name: count, dtype: int64

--- Missing Values Check ---
order_id            0
customer_id         0
order_date        911
order_status        0
ship_city_id     6516
order_total         0
discount_code       0
channel             0
dtype: int64


In [56]:
# 10. Save cleaned dataset
orders.to_csv('./Cleaned_Data/orders_cleaned.csv', index=False)

## Payments data cleaning

In [57]:
# 1. Load data preserving ID formats
payments = pd.read_csv('./Raw_Data/payments.csv', dtype={'payment_id': str, 'order_id': str})

In [58]:
# Standardize headers
payments.columns = payments.columns.str.strip().str.lower()

In [59]:
# 2. Clean string IDs
payments['payment_id'] = payments['payment_id'].astype(str).str.strip()
payments['order_id'] = payments['order_id'].astype(str).str.strip()

In [60]:
# 3. Standardize payment_method
method_mapping = {
    'cod': 'Cash on Delivery',
    'cash on delivery': 'Cash on Delivery',
    'jazzcash': 'JazzCash',
    'easypaisa': 'EasyPaisa',
    'easy paisa': 'EasyPaisa',
    'bank transfer': 'Bank Transfer',
    'debit card': 'Card',
    'card': 'Card'
}

payments['payment_method'] = (
    payments['payment_method']
    .astype(str)
    .str.strip()
    .str.lower()
    .map(lambda x: method_mapping.get(x, x.title()))
)

In [61]:
# 4. Standardize payment_status
status_mapping = {
    'paid': 'Paid',
    'complete': 'Paid',
    'success': 'Paid',
    'cancelled': 'Cancelled',
    'refunded': 'Refunded'
}

payments['payment_status'] = (
    payments['payment_status']
    .astype(str)
    .str.strip()
    .str.lower()
    .map(lambda x: status_mapping.get(x, x.title()))
)

In [62]:
# 5. Clean amount (take absolute value to correct negative records like -1712)
payments['amount'] = (
    payments['amount']
    .astype(str)
    .str.replace(r'[^\d.-]', '', regex=True)
)
payments['amount'] = pd.to_numeric(payments['amount'], errors='coerce').abs()

In [63]:
# 6. Standardize payment_date
payments['payment_date'] = pd.to_datetime(
    payments['payment_date'],
    format='mixed',
    dayfirst=True,
    errors='coerce'
).dt.strftime('%Y-%m-%d')

In [67]:
# Ensure string IDs to preserve leading zeros
orders = pd.read_csv('./Cleaned_Data/orders_cleaned.csv', dtype={'order_id': str})

# 1. Inspect how many missing dates belong to Paid vs Cancelled
print("Missing dates by status before imputation:")
print(payments[payments['payment_date'].isna()]['payment_status'].value_counts())

# 2. Merge order_date from the orders table onto payments
payments = payments.merge(
    orders[['order_id', 'order_date']], 
    on='order_id', 
    how='left'
)

# 3. Impute: If payment_date is NaN AND payment_status is 'Paid', use order_date
mask = payments['payment_date'].isna() & (payments['payment_status'] == 'Paid')
payments.loc[mask, 'payment_date'] = payments.loc[mask, 'order_date']

# 4. Drop the temporary 'order_date' column
payments = payments.drop(columns=['order_date'])


Missing dates by status before imputation:
payment_status
Cancelled    386
Refunded      85
Canceled      32
Paid          26
Pending       24
Refund         7
Name: count, dtype: int64


In [68]:
# 7. Verification Summary
print("--- Payment Methods ---")
print(payments['payment_method'].value_counts(dropna=False))

print("\n--- Payment Status ---")
print(payments['payment_status'].value_counts(dropna=False))

print("\n--- Missing Dates & Amounts ---")
print(payments[['payment_date', 'amount']].isna().sum())

--- Payment Methods ---
payment_method
Cash on Delivery     91592
Card                 28857
EasyPaisa            21112
JazzCash             20776
Bank Transfer        10934
Credit/Debit Card     2037
Jazz Cash             1888
Bank Trf              1046
Name: count, dtype: int64

--- Payment Status ---
payment_status
Paid         161103
Cancelled     12171
Refunded       3089
Canceled        847
Pending         744
Refund          288
Name: count, dtype: int64

--- Missing Dates & Amounts ---
payment_date    582
amount            0
dtype: int64


In [69]:
# 8. Save cleaned file
payments.to_csv('./Cleaned_Data/payments_cleaned.csv', index=False)

## Products Data Cleaning

In [71]:
# 1. Load data
products = pd.read_csv('./Raw_Data/products.csv', dtype={'product_id': str, 'seller_id': str})


In [72]:
# Standardize headers
products.columns = products.columns.str.strip().str.lower()

In [73]:
# 2. Clean string identifier columns
products['product_id'] = products['product_id'].astype(str).str.strip()
products['product_name'] = (
    products['product_name']
    .astype(str)
    .str.strip()
    .str.replace(r'\s+', ' ', regex=True)
)
products['seller_id'] = products['seller_id'].astype(str).str.strip()

In [74]:
# 3. Standardize Categories
category_mapping = {
    'elec.': 'Electronics',
    'electronics': 'Electronics',
    'auto': 'Automotive',
    'automotive': 'Automotive',
    'car accessories': 'Automotive',
    'health & wellness': 'Health & Wellness',
    'health and wellness': 'Health & Wellness',
    'mobile & accessories': 'Mobile & Accessories',
    'mobile and accessories': 'Mobile & Accessories',
    'food & grocery': 'Grocery',
    'grocery': 'Grocery',
    'sports': 'Sports & Outdoors',
    'sports & outdoors': 'Sports & Outdoors',
    'footwear': 'Footwear',
    'fashion': 'Fashion',
    'clothing': 'Fashion',
    'beauty': 'Beauty & Personal Care',
    'beauty & personal care': 'Beauty & Personal Care',
    'toys and games': 'Toys & Games',
    'toys & games': 'Toys & Games',
    'home & kitchen': 'Home & Kitchen'
}

products['category'] = (
    products['category']
    .astype(str)
    .str.strip()
    .str.lower()
    .map(lambda x: category_mapping.get(x, x.title()))
)

In [75]:
# 4. Standardize Sub-Category (clean spacing, fill missing with 'General')
products['sub_category'] = (
    products['sub_category']
    .astype(str)
    .str.strip()
    .replace(['', 'nan', 'None'], np.nan)
)
products['sub_category'] = products['sub_category'].fillna('General')

In [76]:
# 5. Impute missing Brand from product_name
# If brand is blank, extract the first word from product_name
products['brand'] = products['brand'].astype(str).str.strip().replace(['', 'nan', 'None'], np.nan)
inferred_brand = products['product_name'].str.split().str[0]
products['brand'] = products['brand'].fillna(inferred_brand)

In [77]:
# 6. Clean list_price (strip 'Rs.', commas, whitespace)
products['list_price'] = (
    products['list_price']
    .astype(str)
    .str.replace(r'[^\d.]', '', regex=True)
)
products['list_price'] = pd.to_numeric(products['list_price'], errors='coerce')

In [78]:
# 7. Clean cost_price
products['cost_price'] = (
    products['cost_price']
    .astype(str)
    .str.replace(r'[^\d.]', '', regex=True)
)
products['cost_price'] = pd.to_numeric(products['cost_price'], errors='coerce')

In [79]:
# 8. Clean weight_kg (correct negative values using absolute value)
products['weight_kg'] = pd.to_numeric(products['weight_kg'], errors='coerce').abs()

In [80]:
# 9. Verification Summary
print("--- Standardized Categories ---")
print(products['category'].value_counts())

print("\n--- Missing Values Check ---")
print(products.isna().sum())

print("\n--- Sample Cleaned Records ---")
print(products[['product_name', 'category', 'brand', 'list_price', 'weight_kg']].head(10))

--- Standardized Categories ---
category
Fashion                     492
Mobile & Accessories        337
Home & Kitchen              298
Electronics                 291
Beauty & Personal Care      234
Footwear                    198
Sports & Outdoors           185
Grocery                     169
Health & Wellness           156
Toys & Games                135
Automotive                  127
Books & Stationery          115
Home And Kitchen             72
Mobile Acc.                  45
Shoes                        41
Mobiles                      41
Fashion & Apparel            37
Home&Kitchen                 30
Beauty And Personal Care     29
Electronic Items             29
Sports And Outdoors          27
Books                        24
Personal Care                24
Toys                         16
Groceries                    14
Health                       13
Books And Stationery         12
Stationary                    9
Name: count, dtype: int64

--- Missing Values Check ---
product

In [81]:
# 10. Save cleaned dataset
products.to_csv('./Cleaned_Data/products_cleaned.csv', index=False)

## Returns Data Cleaning

In [82]:
# 1. Load data
returns = pd.read_csv('./Raw_Data/returns.csv', dtype={'return_id': str, 'order_item_id': str})

In [83]:
# Clean column headers
returns.columns = returns.columns.str.strip().str.lower()

In [84]:
# 2. Clean string IDs
returns['return_id'] = returns['return_id'].astype(str).str.strip()
returns['order_item_id'] = returns['order_item_id'].astype(str).str.strip()

In [85]:
# 3. Clean return_date (replace invalid patterns like '00/00/2024' with NaN, then standardize)
returns['return_date'] = returns['return_date'].astype(str).str.strip()
returns['return_date'] = returns['return_date'].replace(r'.*00/00.*', np.nan, regex=True)

returns['return_date'] = pd.to_datetime(
    returns['return_date'],
    format='mixed',
    dayfirst=True,
    errors='coerce'
).dt.strftime('%Y-%m-%d')

In [86]:
# 4. Standardize return_reason
reason_mapping = {
    'size/fit issue': 'Size/Fit Issue',
    'size/fit': 'Size/Fit Issue',
    "doesn't fit": 'Size/Fit Issue',
    'damaged product': 'Damaged/Defective',
    'item damaged': 'Damaged/Defective',
    'damaged': 'Damaged/Defective',
    'quality issue': 'Quality Issue',
    'not as described': 'Not As Described',
    'wrong item delivered': 'Wrong Item Delivered',
    'wrong colour': 'Wrong Color/Variant',
    'changed mind': 'Changed Mind',
    'no longer needed': 'Changed Mind',
    'late delivery': 'Late Delivery',
    'missing parts': 'Missing Parts'
}

returns['return_reason'] = (
    returns['return_reason']
    .astype(str)
    .str.strip()
    .str.lower()
    .replace(['', 'nan', 'none'], np.nan)
    .map(lambda x: reason_mapping.get(x, 'Other' if pd.isna(x) else x.title()))
)
returns['return_reason'] = returns['return_reason'].fillna('Not Specified')

In [87]:
# 5. Standardize return_status
status_mapping = {
    'refunded': 'Refunded',
    'refund': 'Refunded',
    'approved': 'Approved',
    'rejected': 'Rejected'
}

returns['return_status'] = (
    returns['return_status']
    .astype(str)
    .str.strip()
    .str.lower()
    .map(lambda x: status_mapping.get(x, x.title()))
)

In [88]:
# 6. Clean refund_amount (convert to numeric float)
returns['refund_amount'] = (
    returns['refund_amount']
    .astype(str)
    .str.replace(r'[^\d.]', '', regex=True)
)
returns['refund_amount'] = pd.to_numeric(returns['refund_amount'], errors='coerce').fillna(0.0)

In [89]:
# 7. Verification Summary
print("--- Cleaned Return Status ---")
print(returns['return_status'].value_counts(dropna=False))

print("\n--- Cleaned Return Reasons ---")
print(returns['return_reason'].value_counts(dropna=False))

print("\n--- Null Dates Check (including repaired 00/00/2024) ---")
print(f"Missing dates: {returns['return_date'].isna().sum()}")

--- Cleaned Return Status ---
return_status
Refunded    11549
Approved     3177
Rejected     1158
Declined      164
Name: count, dtype: int64

--- Cleaned Return Reasons ---
return_reason
Other                     2390
Damaged/Defective         2174
Size/Fit Issue            2148
Not As Described          2046
Quality Issue             1419
Changed Mind              1331
Wrong Item Delivered       536
Colour Mismatch            472
Late Delivery              422
Missing Parts              287
Different From Picture     284
Poor Quality               268
Bad Quality                259
Quality                    240
Size Issue                 239
Wrong Item                 218
Change Of Mind             208
Wrong Size                 203
Changed My Mind            195
Received Wrong Product      96
Color Mismatch              88
Colour Different            86
Delivered Late              83
Too Late                    78
Wrong Color/Variant         74
Late                        65
Parts 

In [90]:
# 8. Save cleaned file
returns.to_csv('./Cleaned_Data/returns_cleaned.csv', index=False)

## Sellers Data Cleaning

In [91]:
# 1. Load data
sellers = pd.read_csv('./Raw_Data/sellers.csv', dtype={'seller_id': str})

In [92]:
# Standardize headers
sellers.columns = sellers.columns.str.strip().str.lower()

In [93]:
# 2. Clean string IDs and seller_name
sellers['seller_id'] = sellers['seller_id'].astype(str).str.strip()
sellers['seller_name'] = (
    sellers['seller_name']
    .astype(str)
    .str.strip()
    .str.replace(r'\s+', ' ', regex=True)
    .str.title()
)

In [94]:
# 3. Standardize join_date
sellers['join_date'] = pd.to_datetime(
    sellers['join_date'],
    format='mixed',
    dayfirst=True,
    errors='coerce'
).dt.strftime('%Y-%m-%d')

In [95]:
# 4. Standardize seller_tier
sellers['seller_tier'] = (
    sellers['seller_tier']
    .astype(str)
    .str.strip()
    .str.title()
    .replace(['', 'Nan', 'None'], np.nan)
)
# Impute missing tier (defaulting unassigned sellers to 'Bronze' or 'Unknown')
sellers['seller_tier'] = sellers['seller_tier'].fillna('Bronze')

In [96]:
# 5. Clean city_id as nullable integer
sellers['city_id'] = pd.to_numeric(sellers['city_id'], errors='coerce').astype('Int64')

In [97]:
# 6. Clean rating
sellers['rating'] = pd.to_numeric(sellers['rating'], errors='coerce')

In [98]:
# Fix out-of-range ratings:
# 11 is a known typo for 1.1 in single-digit precision rating columns
sellers.loc[sellers['rating'] == 11, 'rating'] = 1.1

In [99]:
# If ratings > 5 or <= 0 still exist, coerce them to NaN or clip depending on policy
sellers.loc[(sellers['rating'] > 5.0) | (sellers['rating'] <= 0.0), 'rating'] = np.nan

In [100]:
# 7. Verification Summary
print("--- Seller Tiers ---")
print(sellers['seller_tier'].value_counts(dropna=False))

print("\n--- Rating Summary ---")
print(sellers['rating'].describe())

print("\n--- Missing Values Count ---")
print(sellers.isna().sum())

--- Seller Tiers ---
seller_tier
Bronze    281
Silver     99
Gold       40
Name: count, dtype: int64

--- Rating Summary ---
count    376.000000
mean       3.985638
std        0.706017
min        1.100000
25%        3.700000
50%        4.100000
75%        4.400000
max        5.000000
Name: rating, dtype: float64

--- Missing Values Count ---
seller_id       0
seller_name     0
join_date      38
seller_tier     0
city_id         0
rating         44
dtype: int64


In [101]:
# 8. Save cleaned file
sellers.to_csv('./Cleaned_Data/sellers_cleaned.csv', index=False)

## Shipping Data Cleaning

In [102]:
# 1. Load data preserving string IDs
shipping = pd.read_csv('./Raw_Data/shipping.csv', dtype={'shipment_id': str, 'order_id': str})

In [103]:
# Standardize headers
shipping.columns = shipping.columns.str.strip().str.lower()

In [104]:
# 2. Clean string IDs
shipping['shipment_id'] = shipping['shipment_id'].astype(str).str.strip()
shipping['order_id'] = shipping['order_id'].astype(str).str.strip()

In [105]:
# 3. Standardize courier names
courier_mapping = {
    'tcs': 'TCS',
    'tcs express': 'TCS',
    't.c.s': 'TCS',
    'leopards': 'Leopards',
    'postex': 'PostEx',
    'm&p': 'M&P',
    'mnp': 'M&P',
    'trax': 'Trax',
    'blueex': 'BlueEx'
}

shipping['courier'] = (
    shipping['courier']
    .astype(str)
    .str.strip()
    .str.lower()
    .map(lambda x: courier_mapping.get(x, x.title()))
)

In [106]:
# 4. Standardize shipped_date and delivered_date to YYYY-MM-DD
for col in ['shipped_date', 'delivered_date']:
    shipping[col] = pd.to_datetime(
        shipping[col],
        format='mixed',
        dayfirst=True,
        errors='coerce'
    ).dt.strftime('%Y-%m-%d')

In [107]:
# 5. Clean shipping_fee (numeric float, fill missing with median or 0)
shipping['shipping_fee'] = (
    shipping['shipping_fee']
    .astype(str)
    .str.replace(r'[^\d.]', '', regex=True)
)
shipping['shipping_fee'] = pd.to_numeric(shipping['shipping_fee'], errors='coerce')
# Fill missing fees with the median fee of the respective courier
shipping['shipping_fee'] = shipping.groupby('courier')['shipping_fee'].transform(
    lambda grp: grp.fillna(grp.median())
)

In [108]:
# 6. Standardize delivery_status
status_mapping = {
    'deliverd': 'Delivered',
    'delivered': 'Delivered',
    'completed': 'Delivered',
    'rto': 'RTO',
    'returned': 'RTO',
    'cancelled': 'Cancelled'
}

shipping['delivery_status'] = (
    shipping['delivery_status']
    .astype(str)
    .str.strip()
    .str.lower()
    .map(lambda x: status_mapping.get(x, x.upper() if len(x) <= 3 else x.title()))
)

In [109]:
# 7. Verification Summary
print("--- Couriers ---")
print(shipping['courier'].value_counts())

print("\n--- Delivery Status ---")
print(shipping['delivery_status'].value_counts())

print("\n--- Missing Dates & Fees ---")
print(shipping[['shipped_date', 'delivered_date', 'shipping_fee']].isna().sum())

--- Couriers ---
courier
TCS                45402
Leopards           31088
M&P                21202
Trax               19584
PostEx             18070
BlueEx             10458
Leopard Courier     2264
M & P               1537
Blue Ex              825
Blue-Ex              798
Name: count, dtype: int64

--- Delivery Status ---
delivery_status
Delivered             136793
Complete                7192
Returned To Seller      4899
In Transit              1201
Return To Seller         497
RTO                      494
In-Transit                78
Shipped                   74
Name: count, dtype: int64

--- Missing Dates & Fees ---
shipped_date       6762
delivered_date    19601
shipping_fee          0
dtype: int64


In [110]:
# 8. Save cleaned file
shipping.to_csv('./Cleaned_Data/shipping_cleaned.csv', index=False)